1. Select three datasets to perform your analysis on. This analysis uses Adult, German, and COMPAS datasets from AIF360.


In [1]:
# Install and import datasets
!pip install aif360 # Package for pre-processing/ reweighing

import os
import urllib.request
import aif360
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from aif360.datasets import AdultDataset, GermanDataset, CompasDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing
from IPython.display import Markdown, display

np.random.seed(0) # Pick a seed, so the results stay the same

# Downloading aif360 datasets, so they can be loaded
base = os.path.join(os.path.dirname(aif360.__file__), "data", "raw")
files = {
    "adult":  ["https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
               "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
               "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names"],
    "german": ["https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
               "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.doc"],
    "compas": ["https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"],
}
for name, urls in files.items():
    folder = os.path.join(base, name)
    os.makedirs(folder, exist_ok=True)
    for url in urls:
      urllib.request.urlretrieve(url, os.path.join(folder, url.split("/")[-1]))

# Load datasets

# Adult Census Income
adult_dataset_orig = AdultDataset(
    protected_attribute_names=['sex'],           # This dataset also contains protected attribute for "race" which we do not consider in this evaluation
    privileged_classes=[['Male']],               # Sex = male is considered privileged
    features_to_drop=['race']                    # Ignore race-related attributes
)
## Favoured (privileged) and deprived (unprivileged) groups
adult_privileged_groups = [{'sex': 1}]
adult_unprivileged_groups = [{'sex': 0}]

# German Credit Data
german_dataset_orig = GermanDataset(
    protected_attribute_names=['age'],           # This dataset also contains protected attribute for "sex" which we do not consider in this evaluation
    privileged_classes=[lambda x: x >= 25],      # Age >= 25 is considered privileged
    features_to_drop=['personal_status', 'sex']  # Ignore sex-related attributes
)
## Favoured (privileged) and deprived (unprivileged) groups
german_privileged_groups = [{'age': 1}]
german_unprivileged_groups = [{'age': 0}]

# COMPAS Recidivism Racial Bias
compas_dataset_orig = CompasDataset(
    protected_attribute_names=['sex'],           # This dataset also contains protected attribute for "race" which we do not consider in this evaluation
    privileged_classes=[['Female']],             # Sex = female is considered privileged (females get the favourable label (no reoffending) more often)
    features_to_drop=['race']                    # Ignore race-related attributes
)
## Favoured (privileged) and deprived (unprivileged) groups
compas_privileged_groups = [{'sex': 1}]
compas_unprivileged_groups = [{'sex': 0}]

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


2. Compute the demographic parity difference (DP_Diff), the disparate impact ratio (DIR), and the consistency from the original labels of the dataset before mitigation and training any classifiers

In [2]:
# List with datasets and their groupings for the for-loop
datasets = [
    ("Adult",  adult_dataset_orig,  adult_privileged_groups,  adult_unprivileged_groups),
    ("German", german_dataset_orig, german_privileged_groups, german_unprivileged_groups),
    ("COMPAS", compas_dataset_orig, compas_privileged_groups, compas_unprivileged_groups),
]

# Empty list to save results
results = []

# For loop to calculate DP_diff, DIR and consistency
for name, dataset, privileged_group, unprivileged_group in datasets:
    metric_orig = BinaryLabelDatasetMetric(dataset,
                                           unprivileged_groups = unprivileged_group,
                                           privileged_groups = privileged_group)
    results.append({
        "dataset": name,
        "stage": "before (original labels)",         # Add stage so we can compare the results after reweighing
        "DP_diff": -metric_orig.mean_difference(),   # Demographic (statistical) parity difference
                                                     # (AIF360 calculates unprivileged - privileged; flipped because Kamiran & Calders (2012) uses privileged - unprivileged)
        "DIR": metric_orig.disparate_impact(),       # Disparate impact ratio
        "Consistency": metric_orig.consistency()[0],
    })

results_df = pd.DataFrame(results)
display(results_df)

,dataset,stage,DP_diff,DIR,Consistency
0,Adult,before (original labels),0.198901,0.363470,0.768166
1,German,before (original labels),0.128550,0.821248,0.681400
2,COMPAS,before (original labels),0.127687,0.802925,0.674201


3. Perform a train/test split with stratification.

In [3]:
# AIF360 datasets can't be passed to train_test_split directly due to formatting, therefore the row numbers are split instead and then selected using .subset()
# Adult:
train, test = train_test_split(range(len(adult_dataset_orig.labels)),                       # Row numbers
                                       test_size=0.3,                                       # Same size as AIF360 example
                                       stratify=adult_dataset_orig.labels.ravel(),          # Labels need to have a flat shape for stratification
                                       random_state=0)                                      # Set seed to 0 to get the same split every time
adult_train, adult_test = adult_dataset_orig.subset(train), adult_dataset_orig.subset(test) # Separate the test and train set

# German:
train, test = train_test_split(range(len(german_dataset_orig.labels)),                          # Row numbers
                                       test_size=0.3,                                           # Same size as AIF360 example
                                       stratify=german_dataset_orig.labels.ravel(),             # Labels need to have a flat shape for stratification
                                       random_state=0)                                          # Set seed to 0 to get the same split every time
german_train, german_test = german_dataset_orig.subset(train), german_dataset_orig.subset(test) # Separate the test and train set

# Compas:
train, test = train_test_split(range(len(compas_dataset_orig.labels)),                          # Row numbers
                                       test_size=0.3,                                           # Same size as AIF360 example
                                       stratify=compas_dataset_orig.labels.ravel(),             # Labels need to have a flat shape for stratification
                                       random_state=0)                                          # Set seed to 0 to get the same split every time
compas_train, compas_test = compas_dataset_orig.subset(train), compas_dataset_orig.subset(test) # Separate the test and train set


4. Follow the appropriate procedure for the selected method: pre-processing methods modify training data, in-processing methods are used during model training, and post-processing methods modify predictions after training.

In [4]:
# Reweighing the data
def reweighing(dataset, unprivileged_groups, privileged_groups):
    RW = Reweighing(unprivileged_groups=unprivileged_groups,
                    privileged_groups=privileged_groups)
    dataset_transf = RW.fit_transform(dataset)
    return dataset_transf

# Do NOT reweigh test data, only the training data
adult_train_rw = reweighing(adult_train,  adult_unprivileged_groups,  adult_privileged_groups)
german_train_rw = reweighing(german_train, german_unprivileged_groups, german_privileged_groups)
compas_train_rw = reweighing(compas_train, compas_unprivileged_groups, compas_privileged_groups)

In [5]:
# Training logistic regression classifier and predictions
def log_reg_classifier(train_rw, dataset_test):
    # Scale features on training and test data, fit training data only and flatten labels
    scale_orig = StandardScaler()
    X_train = scale_orig.fit_transform(train_rw.features)
    X_test = scale_orig.transform(dataset_test.features)
    y_train = train_rw.labels.ravel()
    w_train = train_rw.instance_weights.ravel()           # weights of training data

    # Train logistic regression classifier with weights of the training data
    lmodel = LogisticRegression()
    lmodel.fit(X_train, y_train, sample_weight = w_train)

    # Predict on test set
    y_test_pred = lmodel.predict(X_test)

    # Copy the test set and insert the predictions
    test_pred = dataset_test.copy(deepcopy = True) # Deepcopy so labels don't change
    test_pred.labels = y_test_pred.reshape(-1,1)

    return test_pred

# Logistic regression classifier and predictions per dataset
adult_test_pred = log_reg_classifier(adult_train_rw, adult_test)
german_test_pred = log_reg_classifier(german_train_rw, german_test)
compas_test_pred = log_reg_classifier(compas_train_rw, compas_test)

# Baseline predictions using the same classifier on the original training data that was not reweighed
adult_test_pred_bl  = log_reg_classifier(adult_train,  adult_test)
german_test_pred_bl = log_reg_classifier(german_train, german_test)
compas_test_pred_bl = log_reg_classifier(compas_train, compas_test)

5. On the held-out test data, report the demographic parity difference (DP_Diff), the disparate impact ratio (DIR), the consistency, and at least one predictive-performance measure from the final predictions after applying the complete mitigation pipeline.

In [6]:
# List with datasets and their groupings for the for-loop
datasets_pred = [
    ("Adult",  adult_test_pred_bl,  adult_test,  adult_privileged_groups,  adult_unprivileged_groups,  "Baseline, no reweighing (test predictions)"),
    ("German", german_test_pred_bl, german_test, german_privileged_groups, german_unprivileged_groups, "Baseline, no reweighing (test predictions)"),
    ("COMPAS", compas_test_pred_bl, compas_test, compas_privileged_groups, compas_unprivileged_groups, "Baseline, no reweighing (test predictions)"),
    ("Adult",  adult_test_pred,       adult_test,  adult_privileged_groups,  adult_unprivileged_groups,  "After reweighing (test predictions)"),
    ("German", german_test_pred,      german_test, german_privileged_groups, german_unprivileged_groups, "After reweighing (test predictions)"),
    ("COMPAS", compas_test_pred,      compas_test, compas_privileged_groups, compas_unprivileged_groups, "After reweighing (test predictions)"),
]

# For loop to calculate DP_diff, DIR, consistency and accuracy
for name, pred, test_true, privileged_group, unprivileged_group, stage in datasets_pred:
    metric_pred = BinaryLabelDatasetMetric(pred,
                                           unprivileged_groups = unprivileged_group,
                                           privileged_groups = privileged_group)
    y_true = test_true.labels.ravel()
    y_pred = pred.labels.ravel()
    results.append({
        "dataset": name,
        "stage": stage,                                              # Baseline and after reweighing stages, so we can compare the results before reweighing
        "DP_diff": -metric_pred.mean_difference(),                   # Demographic (statistical) parity difference
                                                                     # (AIF360 calculates unprivileged - privileged; flipped because Kamiran & Calders (2012) uses privileged - unprivileged)
        "DIR": metric_pred.disparate_impact(),                       # Disparate impact ratio
        "Consistency": metric_pred.consistency()[0],
        "Accuracy": accuracy_score(y_true, y_pred),                  # predictive-performance measure
        "Balanced accuracy":balanced_accuracy_score(y_true, y_pred), # predictive-performance measure
    })

6. Compare the values of the fairness measures before and after the bias mitigation.

In [7]:
results_df = pd.DataFrame(results)
# Put datasets and stages in the correct order instead of alphabetical
dataset_order = ["Adult", "German", "COMPAS"]
results_df["dataset"] = pd.Categorical(results_df["dataset"], categories=dataset_order, ordered=True)

stage_order = ["before (original labels)",
               "Baseline, no reweighing (test predictions)",
               "After reweighing (test predictions)"]
results_df["stage"] = pd.Categorical(results_df["stage"], categories=stage_order, ordered=True)

# Sort results by dataset and stage, reset index to be in the new order, and round results to 3 decimals
results_df = results_df.sort_values(by=["dataset", "stage"]).reset_index(drop=True).round(3)
display(results_df)
results_df.to_csv("results.csv", index=False)

,dataset,stage,DP_diff,DIR,Consistency,Accuracy,Balanced accuracy
0,Adult,before (original labels),0.199,0.363,0.768,NaN,NaN
1,Adult,"Baseline, no reweighing (test predictions)",0.179,0.318,0.804,0.847,0.765
2,Adult,After reweighing (test predictions),0.083,0.619,0.816,0.839,0.746
3,German,before (original labels),0.129,0.821,0.681,NaN,NaN
4,German,"Baseline, no reweighing (test predictions)",0.045,0.945,0.731,0.753,0.659
5,German,After reweighing (test predictions),-0.008,1.010,0.728,0.740,0.646
6,COMPAS,before (original labels),0.128,0.803,0.674,NaN,NaN
7,COMPAS,"Baseline, no reweighing (test predictions)",0.254,0.687,0.868,0.677,0.669
8,COMPAS,After reweighing (test predictions),0.064,0.902,0.866,0.667,0.659
